In [2]:
"""
BB84 양자키분배 프로토콜 시뮬레이션

양자암호(Quantum Key Distribution)의 대표적인 BB84 프로토콜을
클래식 컴퓨터에서 시뮬레이션하는 교육용 프로그램입니다.

※ 실제 양자역학적 광자를 사용하지 않고, 확률적으로 동작을 모사합니다.
"""

import random
from typing import List, Tuple, Optional


# ============ 기저 정의 ============
# 직선 기저 (Rectilinear): |0⟩, |1⟩ → 수직/수평 편광
# 대각 기저 (Diagonal): |+⟩, |-⟩ → ±45° 편광

RECTILINEAR = 0  # 직선 기저 (+)
DIAGONAL = 1     # 대각 기저 (×)

BASIS_NAMES = {RECTILINEAR: "직선(+)", DIAGONAL: "대각(×)"}


def encode_bit(basis: int, bit: int) -> str:
    """
    비트를 주어진 기저로 인코딩 (양자 상태 표현).
    반환: "V"(수직), "H"(수평), "R"(+45°), "L"(-45°)
    """
    if basis == RECTILINEAR:
        return "V" if bit == 0 else "H"  # 수직/수평
    else:  # DIAGONAL
        return "R" if bit == 0 else "L"  # +45° / -45°


def measure(state: str, basis: int) -> int:
    """
    주어진 기저로 '양자 상태'를 측정하여 비트(0 또는 1)를 반환.
    기저가 맞으면 원래 비트, 틀리면 50% 확률로 0 또는 1.
    """
    # 상태 → 비트 매핑
    state_to_bit = {"V": 0, "H": 1, "R": 0, "L": 1}

    if basis == RECTILINEAR:
        if state in ("V", "H"):
            return state_to_bit[state]  # 기저 일치
        else:
            return random.randint(0, 1)  # 기저 불일치 → 랜덤
    else:  # DIAGONAL
        if state in ("R", "L"):
            return state_to_bit[state]  # 기저 일치
        else:
            return random.randint(0, 1)  # 기저 불일치 → 랜덤


def state_after_eve_measurement(original_state: str, eve_basis: int) -> str:
    """
    Eve가 측정하면 상태가 교란됨.
    Eve가 맞는 기저를 쓰면 그대로, 틀린 기저를 쓰면 50%로 다른 상태로 붕괴.
    """
    if eve_basis == RECTILINEAR:
        if original_state in ("V", "H"):
            return original_state
        else:
            return random.choice(["V", "H"])
    else:
        if original_state in ("R", "L"):
            return original_state
        else:
            return random.choice(["R", "L"])


# ============ BB84 프로토콜 시뮬레이션 ============

def run_bb84(
    num_bits: int = 20,
    eavesdrop: bool = False,
    verbose: bool = True
) -> Tuple[List[int], List[int], Optional[float]]:
    """
    BB84 프로토콜을 시뮬레이션합니다.

    Parameters:
        num_bits: Alice가 전송할 비트 수
        eavesdrop: True면 Eve가 도청 시도
        verbose: True면 단계별 출력

    Returns:
        (alice_key, bob_key, error_rate) - 기저 일치 후 원시 키, 오류율
    """
    random.seed()

    # 1. Alice: 랜덤 비트 및 기저 생성
    alice_bits = [random.randint(0, 1) for _ in range(num_bits)]
    alice_bases = [random.randint(0, 1) for _ in range(num_bits)]

    if verbose:
        print("=" * 60)
        print("BB84 양자키분배 시뮬레이션")
        print("=" * 60)
        print("\n[1단계] Alice가 비트와 기저를 랜덤하게 생성")
        print("  Alice 비트:", " ".join(str(b) for b in alice_bits))
        print("  Alice 기저:", " ".join(BASIS_NAMES[b] for b in alice_bases))

    # 2. 인코딩 및 전송 (Eve 도청 여부에 따라)
    states = []
    for i in range(num_bits):
        s = encode_bit(alice_bases[i], alice_bits[i])
        if eavesdrop:
            eve_basis = random.randint(0, 1)
            s = state_after_eve_measurement(s, eve_basis)
        states.append(s)

    if verbose and eavesdrop:
        print("\n[경고] Eve가 도청했습니다! (각 광자를 랜덤 기저로 측정)")

    # 3. Bob: 랜덤 기저로 측정
    bob_bases = [random.randint(0, 1) for _ in range(num_bits)]
    bob_bits = [measure(states[i], bob_bases[i]) for i in range(num_bits)]

    if verbose:
        print("\n[2단계] Bob이 랜덤 기저로 측정")
        print("  Bob 기저:  ", " ".join(BASIS_NAMES[b] for b in bob_bases))
        print("  Bob 결과:  ", " ".join(str(b) for b in bob_bits))

    # 4. 기저 공개 및 시프팅 (Sifting)
    alice_key = []
    bob_key = []
    for i in range(num_bits):
        if alice_bases[i] == bob_bases[i]:
            alice_key.append(alice_bits[i])
            bob_key.append(bob_bits[i])

    if verbose:
        print("\n[3단계] 기저 비교 - 일치하는 비트만 키로 보유")
        print("  Alice 키:", " ".join(str(b) for b in alice_key))
        print("  Bob 키:  ", " ".join(str(b) for b in bob_key))

    # 5. 오류율 계산
    errors = sum(1 for a, b in zip(alice_key, bob_key) if a != b)
    error_rate = errors / len(alice_key) if alice_key else 0

    if verbose:
        print("\n[결과]")
        print(f"  기저가 일치한 비트 수: {len(alice_key)} / {num_bits}")
        print(f"  오류 수: {errors}")
        print(f"  오류율: {error_rate:.1%}")
        if eavesdrop and error_rate > 0.1:
            print("  ※ 오류율이 높음 → 도청 의심! 키 폐기 권장")

    return alice_key, bob_key, error_rate


def run_statistics(num_trials: int = 1000, bits_per_trial: int = 50):
    """
    BB84를 여러 번 실행하여 통계를 수집합니다.
    - 도청 없음: 오류율 ≈ 0%
    - 도청 있음: 오류율 ≈ 25% (이론값)
    """
    print("\n" + "=" * 60)
    print("BB84 통계 시뮬레이션")
    print("=" * 60)

    # 도청 없음
    errors_no_eve = 0
    total_bits_no_eve = 0
    for _ in range(num_trials):
        ak, bk, _ = run_bb84(bits_per_trial, eavesdrop=False, verbose=False)
        total_bits_no_eve += len(ak)
        errors_no_eve += sum(1 for a, b in zip(ak, bk) if a != b)

    rate_no_eve = errors_no_eve / total_bits_no_eve if total_bits_no_eve else 0
    print(f"\n[도청 없음] {num_trials}회 시행")
    print(f"  총 키 비트: {total_bits_no_eve}, 오류: {errors_no_eve}")
    print(f"  평균 오류율: {rate_no_eve:.2%}")

    # 도청 있음
    errors_with_eve = 0
    total_bits_with_eve = 0
    for _ in range(num_trials):
        ak, bk, _ = run_bb84(bits_per_trial, eavesdrop=True, verbose=False)
        total_bits_with_eve += len(ak)
        errors_with_eve += sum(1 for a, b in zip(ak, bk) if a != b)

    rate_with_eve = errors_with_eve / total_bits_with_eve if total_bits_with_eve else 0
    print(f"\n[도청 있음] {num_trials}회 시행")
    print(f"  총 키 비트: {total_bits_with_eve}, 오류: {errors_with_eve}")
    print(f"  평균 오류율: {rate_with_eve:.2%}")
    print("\n※ 이론: Eve 도청 시 기저가 맞은 비트만 정확하고, 틀린 기저로 측정한 비트는")
    print("  Bob 측 결과의 약 50%가 오류. 전체적으로 ~25% 오류율 기대.")


# ============ 메인 ============

if __name__ == "__main__":
    print("BB84 양자키분배 프로토콜 - 교육용 시뮬레이션")
    print("(양자암호 강의 실습)\n")

    # 예제 1: 도청 없음
    print("\n>>> 예제 1: 정상 통신 (도청 없음)")
    run_bb84(num_bits=16, eavesdrop=False, verbose=True)

    # 예제 2: 도청 있음
    print("\n>>> 예제 2: Eve가 도청하는 경우")
    run_bb84(num_bits=16, eavesdrop=True, verbose=True)

    # 예제 3: 통계
    run_statistics(num_trials=500, bits_per_trial=40)


BB84 양자키분배 프로토콜 - 교육용 시뮬레이션
(양자암호 강의 실습)


>>> 예제 1: 정상 통신 (도청 없음)
BB84 양자키분배 시뮬레이션

[1단계] Alice가 비트와 기저를 랜덤하게 생성
  Alice 비트: 0 0 0 1 0 1 0 0 1 1 1 0 0 1 1 0
  Alice 기저: 대각(×) 직선(+) 대각(×) 대각(×) 직선(+) 대각(×) 직선(+) 직선(+) 직선(+) 대각(×) 직선(+) 직선(+) 직선(+) 대각(×) 대각(×) 직선(+)

[2단계] Bob이 랜덤 기저로 측정
  Bob 기저:   대각(×) 직선(+) 대각(×) 대각(×) 직선(+) 직선(+) 대각(×) 직선(+) 대각(×) 직선(+) 대각(×) 대각(×) 대각(×) 대각(×) 직선(+) 직선(+)
  Bob 결과:   0 0 0 1 0 1 0 0 1 1 1 0 0 1 0 0

[3단계] 기저 비교 - 일치하는 비트만 키로 보유
  Alice 키: 0 0 0 1 0 0 1 0
  Bob 키:   0 0 0 1 0 0 1 0

[결과]
  기저가 일치한 비트 수: 8 / 16
  오류 수: 0
  오류율: 0.0%

>>> 예제 2: Eve가 도청하는 경우
BB84 양자키분배 시뮬레이션

[1단계] Alice가 비트와 기저를 랜덤하게 생성
  Alice 비트: 1 1 1 0 0 0 1 0 1 1 0 0 1 0 1 1
  Alice 기저: 직선(+) 직선(+) 직선(+) 직선(+) 대각(×) 직선(+) 직선(+) 직선(+) 직선(+) 대각(×) 직선(+) 대각(×) 대각(×) 대각(×) 직선(+) 직선(+)

[경고] Eve가 도청했습니다! (각 광자를 랜덤 기저로 측정)

[2단계] Bob이 랜덤 기저로 측정
  Bob 기저:   직선(+) 대각(×) 대각(×) 직선(+) 대각(×) 대각(×) 직선(+) 직선(+) 직선(+) 직선(+) 직선(+) 대각(×) 직선(+) 대각(×) 직선(+) 직선(+)
  Bob 결과:   1 0 1 0 1 1 1 0 1 0 0 0